In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import glob
import torch
from tqdm.auto import tqdm
from datasets import load_dataset

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"

os.chdir("/content/drive/MyDrive/llm-v1")

files = sorted(glob.glob("fine_web/data/*.parquet"))

ds = load_dataset("parquet", data_files=files, split="train")

output_dir = "/content/drive/MyDrive/llm-v1/fine_web/chunks"
os.makedirs(output_dir, exist_ok=True)

chunk_size = 1_000_000
buffer = []
chunk_idx = 0

for row in tqdm(ds["values"], desc="Processing"):
    buffer.extend(row)
    if len(buffer) >= chunk_size:
        tensor = torch.tensor(buffer[:chunk_size], dtype=torch.long)
        torch.save(tensor, f"{output_dir}/chunk_{chunk_idx}.pt")
        buffer = buffer[chunk_size:]
        chunk_idx += 1

if buffer:
    torch.save(torch.tensor(buffer, dtype=torch.long),
               f"{output_dir}/chunk_{chunk_idx}.pt")

Mounted at /content/drive


Resolving data files:   0%|          | 0/199 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/161 [00:00<?, ?it/s]

Processing:   0%|          | 0/9953990 [00:00<?, ?it/s]